# Задание 1

Для каждого покупателя получите таблицу со столбцами:
 • preferred_payment - самый частый метод оплаты (при ничьей зафиксируйте правило, например lexicographically первый);
 • total_spent - сумма всех трат;
 • addons_spent - сумма на дополнительные услуги и аксессуары;
 • orders_count - число заказов.
 
 Отсортируйте по total_spent по убыванию и выведите топ-10.

Пример вывода (формат; числа зависят от датасета):

   customer_id  preferred_payment  total_spent  addons_spent  orders_count
 0         C042     Credit Card              15420.50        890.00             7
 1         C018     PayPal                     14810.00        450.00             5
 2         C103     Debit Card               13990.25       1200.00             6
 ...

Пограничные случаи:

• Покупатель с одним заказом - orders_count = 1, preferred_payment = единственный метод.
 • Два метода оплаты с одинаковой частотой - выберите один по зафиксированному правилу (укажите его в комментарии к коду).
 • Покупатель без доп. услуг/аксессуаров - addons_spent = 0.
 • Пропуски в сумме или методе оплаты - обработайте (отфильтруйте или заполните) и кратко опишите решение в комментарии.

In [1]:
import pandas as pd

In [9]:
df = pd.read_csv('Electronic_sales_Sep2023-Sep2024.csv')
df.head(10)

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56
5,1004,41,Female,No,Smartphone,SKU1001,5,Completed,Credit Card,83.00,20.75,4,2024-05-26,Standard,"Impulse Item,Accessory",65.78
6,1005,25,Female,No,Smartwatch,SKU1003,3,Completed,Paypal,7603.47,844.83,9,2024-01-30,Overnight,NaN,0.00
7,1005,25,Female,No,Laptop,SKU1005,3,Completed,Debit Card,4175.64,463.96,9,2024-06-24,Overnight,"Extended Warranty,Extended Warranty",75.33
8,1006,24,Male,No,Smartphone,SKU1004,2,Cancelled,Debit Card,5538.33,791.19,7,2023-10-03,Standard,Impulse Item,43.05
9,1006,24,Male,Yes,Laptop,SKU1005,3,Completed,Cash,4175.64,463.96,9,2024-01-01,Express,NaN,0.00


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer ID        20000 non-null  int64  
 1   Age                20000 non-null  int64  
 2   Gender             19999 non-null  str    
 3   Loyalty Member     20000 non-null  str    
 4   Product Type       20000 non-null  str    
 5   SKU                20000 non-null  str    
 6   Rating             20000 non-null  int64  
 7   Order Status       20000 non-null  str    
 8   Payment Method     20000 non-null  str    
 9   Total Price        20000 non-null  float64
 10  Unit Price         20000 non-null  float64
 11  Quantity           20000 non-null  int64  
 12  Purchase Date      20000 non-null  str    
 13  Shipping Type      20000 non-null  str    
 14  Add-ons Purchased  15132 non-null  str    
 15  Add-on Total       20000 non-null  float64
dtypes: float64(3), int64(4), str(9)
m

In [13]:
df['Payment Method'].unique()

<ArrowStringArray>
['Credit Card', 'Paypal', 'Cash', 'Debit Card', 'Bank Transfer', 'PayPal']
Length: 6, dtype: str

In [15]:
df['Add-on Total'] = df['Add-on Total'].fillna(0)
df['Payment Method'] = df['Payment Method'].replace({'Paypal' : 'PayPal'})

In [16]:
df['Payment Method'].unique()

<ArrowStringArray>
['Credit Card', 'PayPal', 'Cash', 'Debit Card', 'Bank Transfer']
Length: 5, dtype: str

In [17]:
result = pd.DataFrame({'customer_id': df['Customer ID'].unique()})
result.head()

,customer_id
0,1000
1,1002
2,1003
3,1004
4,1005


In [19]:
pref_payment = df.groupby('Customer ID')['Payment Method'].agg(lambda x: sorted(x.mode())[0])
# выбираю лексически первый метод
pref_payment.head()

Customer ID
1000    Credit Card
1002           Cash
1003           Cash
1004    Credit Card
1005     Debit Card
Name: Payment Method, dtype: str

In [20]:
result['preferred_payment'] = result['customer_id'].map(pref_payment)
result.head()

,customer_id,preferred_payment
0,1000,Credit Card
1,1002,Cash
2,1003,Cash
3,1004,Credit Card
4,1005,Debit Card


In [22]:
total_spent = df.groupby('Customer ID')['Total Price'].sum()
result['total_spent'] = result['customer_id'].map(total_spent)

addons_spent = df.groupby('Customer ID')['Add-on Total'].sum()
result['addons_spent'] = result['customer_id'].map(addons_spent)

result.head()


,customer_id,preferred_payment,total_spent,addons_spent
0,1000,Credit Card,6279.42,66.30
1,1002,Cash,5020.60,60.16
2,1003,Cash,41.50,35.56
3,1004,Credit Card,83.00,65.78
4,1005,Debit Card,11779.11,75.33


In [23]:
orders_count = df.groupby('Customer ID').size()
result['orders_count'] = result['customer_id'].map(orders_count)
result.head()

,customer_id,preferred_payment,total_spent,addons_spent,orders_count
0,1000,Credit Card,6279.42,66.30,2
1,1002,Cash,5020.60,60.16,2
2,1003,Cash,41.50,35.56,1
3,1004,Credit Card,83.00,65.78,1
4,1005,Debit Card,11779.11,75.33,2


In [24]:
result = result.sort_values(by='total_spent', ascending=False)
result.head(10)

,customer_id,preferred_payment,total_spent,addons_spent,orders_count
9728,16357,Bank Transfer,34563.70,495.19,7
10049,16863,PayPal,33035.92,486.17,5
8060,13813,Credit Card,31830.16,268.64,5
6475,11476,PayPal,31077.61,301.59,5
7028,12276,Bank Transfer,30961.18,329.07,6
7935,13635,Credit Card,30260.36,480.73,5
7352,12749,Credit Card,29394.56,619.43,5
9087,15399,PayPal,29084.88,305.39,3
7054,12319,Bank Transfer,27352.32,226.38,3
12133,19996,Bank Transfer,27296.78,432.12,6
